# microsimflow on Google Colab: Interactive Recipe Builder

This notebook provides a graphical user interface (GUI) to design complex composite microstructures, preview 3D filler placements, and automatically generate batch execution scripts for the `microsimflow` pipeline.

**Hardware Requirement:**
The GUI itself runs smoothly on a standard CPU runtime. A **T4 GPU** is required *only* if you intend to execute the `chfem` solver for actual property calculations in Step 4. (To change, go to `Runtime` > `Change runtime type` > Select `T4 GPU`).

### Step 1: Setup Workspace & Install Dependencies
We clone the `microsimflow` repository, configure the workspace, and set the `PYTHONPATH`. We also install `plotly` and `scikit-image` for robust 3D rendering in Colab.

In [ ]:
# 1. Clone the repository and setup workspace
!git clone https://github.com/hikuram/microsimflow.git /content/microsimflow
!mkdir -p /content/workspace/exp
!cp /content/microsimflow/exp/*.py /content/workspace/exp

# 2. Set PYTHONPATH to include the microsimflow package
import os
os.environ["PYTHONPATH"] = "/content/microsimflow:" + os.environ.get("PYTHONPATH", "")

# 3. Install visualization dependencies
!pip install plotly scikit-image numba -q

### Step 2: Initialize the GUI Application
Run this cell to define the `RecipeBuilderApp`. This interface handles global settings, filler recipes, 3D previews, and python script generation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import textwrap
import itertools

import plotly.graph_objects as go
from skimage.measure import marching_cubes

from microsimflow.micro_builder import (
    build_single_phase_grid, build_tpms_grid_with_target_ratio,
    build_lamellar_grid, build_cylinder_hex_grid, build_bcc_grid,
    build_sea_island_grid, build_island_sea_grid,
    place_fillers_hybrid, place_adaptive_fibers,
    get_rigid_cylinder_mask, get_flake_mask, get_sphere_mask,
    get_irregular_fiber_mask, get_flexible_fiber_mask,
    get_agglomerate_mask, get_staggered_flakes_mask
)

class RecipeBuilderApp:
    def __init__(self):
        self.output = widgets.Output()
        self.preview_log = widgets.Output()

        self.preview_fig = go.FigureWidget()
        self.preview_fig.update_layout(
            scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
            margin=dict(l=0, r=0, b=0, t=30),
            title="Microstructure 3D Preview (Waiting...)",
            height=600
        )

        self.grid_size = widgets.IntText(value=100, description='Grid Size:', layout={'width': '200px'})
        self.voxel_size = widgets.FloatText(value=1e-8, description='Voxel(m):', layout={'width': '200px'})
        self.bg_type = widgets.Dropdown(
            options=['single', 'gyroid', 'lamellar', 'cylinder', 'bcc', 'sea_island', 'island_sea'],
            value='single', description='BG Type:', layout={'width': '200px'}
        )
        self.physics_mode = widgets.Dropdown(
            options=['thermal', 'electrical', 'mechanics'],
            value='electrical', description='Mode:', layout={'width': '200px'}
        )
        self.solver = widgets.Dropdown(
            options=['chfem', 'puma', 'both', 'skip'],
            value='chfem', description='Solver:', layout={'width': '200px'}
        )

        self.recipe_container = widgets.VBox([])
        self.btn_add = widgets.Button(description="Add Filler", icon="plus", button_style="info")
        self.btn_add.on_click(self.add_filler_row)

        self.preview_step_slider = widgets.IntSlider(
            value=0, min=0, max=0, step=1,
            description='Preview Step:',
            layout={'width': '400px'},
            style={'description_width': 'initial'}
        )
        self.step_info = widgets.Label(value="Total combinations: 1")

        self.stretch_min = widgets.FloatText(value=1.0, step=0.1, description='L Min:', layout={'width': '150px'})
        self.stretch_max = widgets.FloatText(value=1.0, step=0.1, description='L Max:', layout={'width': '150px'})
        self.stretch_steps = widgets.IntText(value=1, description='Steps:', layout={'width': '150px'})

        self.btn_preview = widgets.Button(description="Generate 3D Preview", button_style="warning")
        self.btn_generate = widgets.Button(description="Export .py Script", button_style="success")

        self.btn_preview.on_click(self.show_3d_preview)
        self.btn_generate.on_click(self.generate_script)

        self.add_filler_row(None)

    def update_step_count(self, _=None):
        state = self.get_ui_state()
        counts = [f['vf_steps'] for f in state['fillers']]
        total = int(np.prod(counts)) if counts else 1
        self.preview_step_slider.max = max(0, total - 1)
        self.step_info.value = f"Total combinations: {total}"

    def add_filler_row(self, _):
        default_params = {
            'rigidfiber': "length=60:radius=2",
            'flake': "radius=10:thickness=2",
            'sphere': "radius=5",
            'irregfiber': "length=60:shape=bean:radius=5:ratio=0.5",
            'flexfiber': "length=90:radius=2:max_bend_deg=90:max_total_bends=10",
            'agglomerate': "num_fibers=5:length=90:radius=2:max_bend_deg=90",
            'staggered': "radius=15:layer_thickness=2:min_layers=1:max_layers=4:max_offset_pct=30"
        }
        f_type = widgets.Dropdown(options=list(default_params.keys()), layout={'width': '120px'})
        vf_min = widgets.FloatText(value=0.10, step=0.01, description='Vf Min:', layout={'width': '140px'})
        vf_max = widgets.FloatText(value=0.10, step=0.01, description='Vf Max:', layout={'width': '140px'})
        vf_steps = widgets.IntText(value=1, description='Steps:', layout={'width': '120px'})
        params = widgets.Text(value=default_params[f_type.value], layout={'width': '300px'})

        f_type.observe(lambda c: setattr(params, 'value', default_params[c['new']]) if c['name']=='value' else None, names='value')
        vf_steps.observe(self.update_step_count, names='value')

        btn_del = widgets.Button(icon="trash", layout={'width': '40px'}, button_style="danger")
        row = widgets.HBox([f_type, vf_min, vf_max, vf_steps, params, btn_del])

        def delete_me(_):
            self.recipe_container.children = [c for c in self.recipe_container.children if c != row]
            self.update_step_count()

        btn_del.on_click(delete_me)
        self.recipe_container.children += (row,)
        self.update_step_count()

    def get_ui_state(self):
        fillers = []
        for row in self.recipe_container.children:
            fillers.append({
                'type': row.children[0].value, 'vf_min': row.children[1].value, 'vf_max': row.children[2].value,
                'vf_steps': max(1, row.children[3].value), 'params': row.children[4].value
            })
        return {
            'grid_size': self.grid_size.value, 'bg_type': self.bg_type.value,
            'physics_mode': self.physics_mode.value, 'fillers': fillers,
            'voxel_size': self.voxel_size.value, 'solver': self.solver.value,
            'stretch_min': self.stretch_min.value, 'stretch_max': self.stretch_max.value,
            'stretch_steps': self.stretch_steps.value
        }

    def show_3d_preview(self, _):
        with self.preview_log:
            clear_output()
            print("🚀 Starting RSA placement algorithm...")

        self.btn_preview.description = "⏳ Calculating..."
        self.btn_preview.disabled = True

        try:
            state = self.get_ui_state()
            original_size = state['grid_size']
            size = min(original_size, 200)
            if original_size > 200:
                with self.preview_log: print(f"⚠️ Size={original_size} is too large. Using Size=200 for preview.")

            vf_lists = [np.linspace(f['vf_min'], f['vf_max'], f['vf_steps']).tolist() if f['vf_steps'] > 1 else [f['vf_min']] for f in state['fillers']]
            if not vf_lists:
                with self.preview_log: print("⚠️ No fillers added.")
                return

            combos = list(itertools.product(*vf_lists))
            target_combo = combos[self.preview_step_slider.value]

            with self.preview_log:
                print(f"  Previewing Step: {self.preview_step_slider.value}")
                print(f"  Volume Fractions: {target_combo}")

            bg_type = state['bg_type']
            if bg_type == "single": _, tpms_grid, _, _ = build_single_phase_grid(size)
            elif bg_type == "lamellar": _, tpms_grid, _, _ = build_lamellar_grid(size, 0.5)
            elif bg_type == "cylinder": _, tpms_grid, _, _ = build_cylinder_hex_grid(size, 0.5)
            elif bg_type == "bcc": _, tpms_grid, _, _ = build_bcc_grid(size, 0.5)
            elif bg_type == "sea_island": _, tpms_grid, _, _ = build_sea_island_grid(size, 0.5)
            elif bg_type == "island_sea": _, tpms_grid, _, _ = build_island_sea_grid(size, 0.5)
            else: _, tpms_grid, _, _ = build_tpms_grid_with_target_ratio(size, 0.5)

            comp_grid = np.zeros((size, size, size), dtype=np.uint8)
            shell_grid = np.zeros_like(comp_grid)

            filler_id = 4
            for i, f in enumerate(state['fillers']):
                f_type = f['type']
                vf = target_combo[i]
                if vf <= 0: continue

                opts = {}
                for p in f['params'].split(':'):
                    if '=' not in p: continue
                    k, v = p.split('=')
                    try:
                        opts[k] = int(v) if v.isdigit() else float(v)
                    except ValueError:
                        opts[k] = v

                hybrid_args = {
                    'comp_grid': comp_grid, 'tpms_grid': tpms_grid, 'target_vol_frac': vf,
                    'physics_mode': state['physics_mode'], 'shell_count_grid': shell_grid,
                    'filler_id': filler_id, 'inter_id': 3, 'tunnel_radius': 2, 'log_file': None
                }

                with self.preview_log:
                    print(f"  Placing {f_type} ID={filler_id} at Vf={vf:.4f}...")
                    if f_type == "rigidfiber":
                        place_fillers_hybrid(filler_func=get_rigid_cylinder_mask, kwargs={'length': opts.get('length', 60), 'radius': opts.get('radius', 2)}, **hybrid_args)
                    elif f_type == "flake":
                        place_fillers_hybrid(filler_func=get_flake_mask, kwargs={'radius': opts.get('radius', 10), 'thickness': opts.get('thickness', 2)}, **hybrid_args)
                    elif f_type == "sphere":
                        place_fillers_hybrid(filler_func=get_sphere_mask, kwargs={'radius': opts.get('radius', 5)}, **hybrid_args)
                    elif f_type == "irregfiber":
                        place_fillers_hybrid(filler_func=get_irregular_fiber_mask, kwargs={'length': opts.get('length', 60), 'shape_type': opts.get('shape', 'bean'), 'radius_max': opts.get('radius', 5), 'ratio': opts.get('ratio', 0.5)}, **hybrid_args)
                    elif f_type == "flexfiber":
                        place_fillers_hybrid(filler_func=get_flexible_fiber_mask, kwargs={'length': opts.get('length', 90), 'radius': opts.get('radius', 2)}, **hybrid_args)
                    elif f_type == "agglomerate":
                        place_fillers_hybrid(filler_func=get_agglomerate_mask, kwargs={'num_fibers': opts.get('num_fibers', 5), 'length': opts.get('length', 90), 'radius': opts.get('radius', 2)}, **hybrid_args)
                    elif f_type == "staggered":
                        place_fillers_hybrid(filler_func=get_staggered_flakes_mask, kwargs={'radius': opts.get('radius', 15), 'layer_thickness': opts.get('layer_thickness', 2)}, **hybrid_args)
                filler_id += 1

            final_grid = np.where(comp_grid > 0, comp_grid, tpms_grid)
            unique_ids = np.unique(final_grid)
            filler_ids = unique_ids[unique_ids >= 4]

            traces = []
            colors = ['coral', 'royalblue', 'mediumseagreen', 'gold', 'orchid']
            for idx, f_id in enumerate(filler_ids):
                mask = (final_grid == f_id).astype(float)
                verts, faces, _, _ = marching_cubes(mask, level=0.5)
                traces.append(go.Mesh3d(x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], color=colors[idx % len(colors)], name=f"Phase {f_id}"))

            with self.preview_fig.batch_update():
                self.preview_fig.data = []
                for trace in traces: self.preview_fig.add_trace(trace)
                self.preview_fig.layout.scene.xaxis.range = [0, size]
                self.preview_fig.layout.scene.yaxis.range = [0, size]
                self.preview_fig.layout.scene.zaxis.range = [0, size]
                self.preview_fig.layout.title = f"Preview: Step {self.preview_step_slider.value} (Grid={size})"

        finally:
            self.btn_preview.description = "Generate 3D Preview"
            self.btn_preview.disabled = False
            with self.preview_log: print("✅ Preview generation completed.")

    def build_python_code(self, state):
        s_min, s_max, s_steps = state['stretch_min'], state['stretch_max'], state['stretch_steps']
        stretch_ratios = np.linspace(s_min, s_max, s_steps).tolist() if s_steps > 1 else [s_min]
        stretch_str = " ".join([f"{x:.3f}" for x in stretch_ratios])

        vf_arrays_code = []
        for i, f in enumerate(state['fillers']):
            arr = f"np.linspace({f['vf_min']}, {f['vf_max']}, {f['vf_steps']})" if f['vf_steps'] > 1 else f"[{f['vf_min']}]"
            vf_arrays_code.append(f"    vf_sweep_{i} = {arr}")

        vf_arrays_str = "\n".join(vf_arrays_code)
        sweep_vars = ", ".join([f"vf_sweep_{i}" for i in range(len(state['fillers']))])

        recipe_build_code = []
        for i, f in enumerate(state['fillers']):
            recipe_build_code.append(f'                f"{f["type"]}:{{combo[{i}]:.4f}}:{f["params"]}"')

        recipe_list_str = ",\n".join(recipe_build_code)

        template = textwrap.dedent(f"""\
import subprocess
import os
import itertools
import numpy as np

def run_generated_sweep():
    out_dir = "result_gui_sweep"
    os.makedirs(out_dir, exist_ok=True)
    csv_log = "gui_sweep_results.csv"
    print("--- Starting GUI Generated Sweep ---")

{vf_arrays_str}
    stretch_args = ["--stretch_ratios"] + "{stretch_str}".split()
    combinations = list(itertools.product({sweep_vars}))

    for idx, combo in enumerate(combinations):
        recipe = [
{recipe_list_str}
        ]
        basename = os.path.join(out_dir, f"sweep_{{idx:03d}}")

        cmd = [
            "python3", "-m", "run_pipeline",
            "--size", "{state['grid_size']}",
            "--voxel_size", "{state['voxel_size']}",
            "--bg_type", "{state['bg_type']}",
            "--physics_mode", "{state['physics_mode']}",
            "--solver", "{state['solver']}",
            "--basename", basename, "--csv_log", csv_log, "--recipe"
        ] + recipe + stretch_args

        print(f"\\nRunning {{idx+1}}/{{len(combinations)}}...")
        subprocess.run(cmd)

if __name__ == "__main__":
    run_generated_sweep()
""")
        return template

    def generate_script(self, _):
        with self.output:
            clear_output(wait=True)
            state = self.get_ui_state()
            if not state['fillers']:
                print("Error: Please add at least one filler.")
                return
            with open("run_custom_sweep.py", "w") as f:
                f.write(self.build_python_code(state))

            total_cases = np.prod([f['vf_steps'] for f in state['fillers']]) if state['fillers'] else 0
            print(f"✅ 'run_custom_sweep.py' generated successfully. (Total {total_cases} patterns)")

### Step 3: Launch the GUI Interface
Execute this cell to display the application. We use a fixed-height container to prevent the Colab output area from collapsing during rendering updates.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

app = RecipeBuilderApp()

# Fixed-height container to prevent UI jitter in Colab
preview_container = widgets.Box(
    [app.preview_fig],
    layout=widgets.Layout(
        width='100%',
        height='620px',
        border='1px solid #e0e0e0',
        padding='5px'
    )
)

display(widgets.VBox([
    widgets.HTML("<h3>1. Global Settings</h3>"),
    widgets.HBox([app.grid_size, app.voxel_size]),
    widgets.HBox([app.bg_type, app.physics_mode, app.solver]),
    widgets.HTML("<h3>2. Filler Recipe Sweep & Preview Select</h3>"),
    app.recipe_container, app.btn_add,
    widgets.HBox([app.preview_step_slider, app.step_info]),
    widgets.HTML("<h3>3. Deformation Sweep</h3>"),
    widgets.HBox([app.stretch_min, app.stretch_max, app.stretch_steps]),
    widgets.HTML("<hr>"),
    widgets.HBox([app.btn_preview, app.btn_generate]),
    preview_container,
    app.preview_log,
    app.output
]))

### Step 4: Install Additional Dependencies & Compile `chfem` Solver (GPU Required)
**Note:** Run this cell *only* if you selected `chfem` as your solver in the GUI and wish to execute the calculation suite now. This step requires a **T4 GPU** runtime.

In [ ]:
# 1. Verify CUDA compiler
!nvcc --version

# 2. Clone and compile chfem (using the fork with the typo fix)
!git clone https://github.com/hikuram/chfem.git /content/chfem
!cd /content/chfem && mkdir -p build && cd build && cmake .. \
      -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_CUDA_ARCHITECTURES=75 \
      -DCMAKE_C_FLAGS="-DCUDAPCG_MATKEY_32BIT" \
      -DCMAKE_CUDA_FLAGS="-DCUDAPCG_MATKEY_32BIT" && \
    make -j2

# 3. Install Python dependencies
!pip install pyvista numba

import os
# Add compiled chfem to system PATH
os.environ['PATH'] = '/content/chfem:/content/microsimflow:' + os.environ['PATH']

### Step 5: Execute the Simulation
After configuring your structure and generating the script in Step 3, run the cell below to execute the batch calculation in your workspace.

In [ ]:
!python run_custom_sweep.py